# Exploración y preparación de los datos

**Autor:** Mariana Bedoya Arismendy  
**Fecha:** 2026-08-17  
**Incidencia:** 2 - Exploración de los datos

### Descripción:

Este notebook explora y prepara el dataset de pacientes con problemas de hígado. Se revisan su estructura, valores nulos y tipos de datos antes de guardarlo en formato Parquet.


## 📚 Importar Librerías

In [ ]:
# Librerías necesarias para cargar, explorar y guardar los datos.
from pathlib import Path

import pandas as pd
import pyarrow as pa

: 

## 💾 Cargar los datos

In [ ]:
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "data" / "01_raw").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("No se encontró la raíz del proyecto")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "01_raw/Pacientes_porblemas_higado_india.csv"

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"No existe el dataset: {RAW_DATA_PATH}")

pacientes_df = pd.read_csv(RAW_DATA_PATH, low_memory=False, keep_default_na=False)
print(f"Archivo cargado: {RAW_DATA_PATH.relative_to(PROJECT_ROOT)}")
print(
    f"Dimensiones iniciales: {pacientes_df.shape[0]} filas x {pacientes_df.shape[1]} columnas"
)

## 📊 Descripción de los datos

In [ ]:
pacientes_df.info()

In [ ]:
pacientes_df.sample(10, random_state=42)

## Valores nulos

Primero se revisaron las diferentes formas en que aparecen los datos faltantes en el archivo. Antes de hacer la limpieza, se contaron estos valores para identificar qué columnas estaban afectadas.

In [ ]:
missing_tokens = ["", "?", "N/A", "NA", "NaN"]
missing_token_summary = pd.Series(
    {token: int((pacientes_df == token).sum().sum()) for token in missing_tokens},
    name="casos",
)
missing_by_column_before = (
    pacientes_df.iloc[:, :11].isin(missing_tokens).sum().sort_values(ascending=False)
)
print("Representaciones de faltantes encontradas en el archivo:")
print(missing_token_summary)
print("Faltantes por columna clínica antes de la unificación:")
print(missing_by_column_before[missing_by_column_before > 0])

pacientes_df = pacientes_df.replace(missing_tokens, pd.NA)
null_summary = pacientes_df.isna().sum().sort_values(ascending=False)
print("Valores faltantes después de la unificación:")
print(null_summary[null_summary > 0].head(15))

### Hallazgos sobre valores faltantes

En todo el archivo se encontraron **650.444 cadenas vacías**. Este total incluye las 1.013 columnas `Unnamed:*`, por lo que no corresponde únicamente a datos faltantes clínicos. No se encontraron valores `?`, `N/A`, `NA` ni `NaN`.

En las 11 columnas clínicas se encontraron **98 cadenas vacías**: `Age` (1), `Gender` (8), `Total_Bilirubin` (6), `Direct_Bilirubin` (7), `Alkaline_Phosphotase` (17), `Alamine_Aminotransferase` (19), `Aspartate_Aminotransferase` (12), `Total_Protiens` (7), `Albumin` (2), `Albumin_and_Globulin_Ratio` (4) y `Dataset` (15).

Estas cadenas representan datos faltantes y no categorías válidas ni mediciones con valor cero. Se reemplazaron por `pd.NA` para usar una representación común. Después de seleccionar las variables clínicas y convertir sus tipos, se mantienen los mismos **98 valores faltantes**. No se imputan porque en esta etapa solo se identifican y se organizan los datos; no se deben inventar valores clínicos.

## Selección de las columnas clínicas

El archivo contiene 11 variables clínicas y 1.013 columnas adicionales con nombres `Unnamed:*`. Antes de descartarlas, se revisó si tenían contenido y si ese contenido aportaba información nueva.

Después de esta revisión, se conservarán únicamente las 11 columnas clínicas originales.

In [ ]:
clinical_columns = [
    "Age",
    "Gender",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Protiens",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
    "Dataset",
]

auxiliary_columns = [
    column for column in pacientes_df.columns if column.startswith("Unnamed:")
]
auxiliary_non_null = pacientes_df[auxiliary_columns].notna()
auxiliary_non_null_values = int(auxiliary_non_null.sum().sum())
auxiliary_rows_with_values = int(auxiliary_non_null.any(axis=1).sum())
auxiliary_repetitions, auxiliary_remainder = divmod(
    len(auxiliary_columns), len(clinical_columns)
)
auxiliary_pattern_matches = []
for row_index in pacientes_df.index[auxiliary_non_null.any(axis=1)]:
    clinical_values = (
        pacientes_df.loc[row_index, clinical_columns]
        .astype("string")
        .fillna("<NA>")
        .tolist()
    )
    auxiliary_values = (
        pacientes_df.loc[row_index, auxiliary_columns]
        .astype("string")
        .fillna("<NA>")
        .tolist()
    )
    expected_auxiliary_values = (
        clinical_values * auxiliary_repetitions + clinical_values[:auxiliary_remainder]
    )
    auxiliary_pattern_matches.append(auxiliary_values == expected_auxiliary_values)
auxiliary_matching_rows = sum(auxiliary_pattern_matches)
print(f"Columnas auxiliares detectadas: {len(auxiliary_columns)}")
print(f"Valores no nulos en columnas auxiliares: {auxiliary_non_null_values}")
print(f"Filas con algún valor en columnas auxiliares: {auxiliary_rows_with_values}")
print(f"Filas que repiten las variables clínicas: {auxiliary_matching_rows}")
print(f"Repeticiones completas de las 11 variables: {auxiliary_repetitions}")
print("Cantidad de valores auxiliares por fila:")
print(auxiliary_non_null.sum(axis=1).value_counts().sort_index())

missing_columns = sorted(set(clinical_columns) - set(pacientes_df.columns))
if missing_columns:
    raise KeyError(f"Faltan columnas clínicas: {missing_columns}")

initial_column_count = pacientes_df.shape[1]
pacientes_df = pacientes_df.loc[:, clinical_columns].copy()
print(f"Columnas conservadas: {pacientes_df.shape[1]}")
print(
    f"Columnas auxiliares descartadas: {initial_column_count - pacientes_df.shape[1]}"
)
pacientes_df.info()

### Hallazgos sobre columnas `Unnamed:*`

Se identificaron **1.013 columnas** con el prefijo `Unnamed:`. Contienen **21.273 valores no nulos distribuidos en 21 filas**. En esas filas, los valores siguen exactamente el patrón de las 11 variables clínicas: las repiten 92 veces y agregan nuevamente el primer valor.

Como la comparación se cumple en las 21 filas con contenido, estas columnas no aportan variables nuevas. Se descartan y se conservan las 11 columnas clínicas, evitando duplicar la misma información.

### Variables categóricas

- `Gender`: sexo reportado del paciente.
- `Dataset`: clasificación del paciente utilizada como variable objetivo, con valores 1 y 2.


### Variables nominales

`Gender` es una variable nominal y `Dataset` se conserva como categórica para no imponer una interpretación numérica a sus etiquetas.

### Variables numéricas

Las variables clínicas se dividen en dos grupos numéricos: `Age`, que representa la edad en años y se manejará como entero, y las mediciones de bilirrubina, enzimas, proteínas, albúmina y su relación, que se manejarán como números decimales. `Dataset` no es numérica: representa la clase del paciente y se mantiene como categórica.

#### Mediciones clínicas

Las mediciones clínicas se convertirán a tipos numéricos para poder compararlas y analizarlas. Los valores faltantes se conservarán durante la conversión.

### Variable objetivo

`Dataset` contiene las clases 1 y 2. Se conservará como categórica porque sus valores son etiquetas de clasificación y no representan simplemente verdadero o falso.

## Variables de texto

`Gender` contiene el sexo reportado del paciente. Se lee como texto y se convertirá a una variable categórica durante la preparación.

## Convertir tipos de datos

Cada columna se convertirá al tipo de dato que corresponde a su contenido. Los valores faltantes se conservarán y no se reemplazarán por valores inventados.

In [ ]:
cols_categoric = ["Gender", "Dataset"]
unexpected_gender = sorted(set(pacientes_df["Gender"].dropna()) - {"Male", "Female"})
unexpected_dataset = sorted(set(pacientes_df["Dataset"].dropna()) - {"1", "2"})
print(f"Valores inesperados en Gender: {unexpected_gender}")
print(f"Valores inesperados en Dataset: {unexpected_dataset}")

pacientes_df["Gender"] = pacientes_df["Gender"].astype("category")
pacientes_df["Dataset"] = pd.Categorical(pacientes_df["Dataset"], categories=["1", "2"])

In [ ]:
print(pacientes_df[cols_categoric].dtypes)
print(pacientes_df["Gender"].value_counts(dropna=False))
print(pacientes_df["Dataset"].value_counts(dropna=False).sort_index())

### Hallazgos sobre variables categóricas

No se encontraron categorías inesperadas entre los valores no nulos: `Gender` solo contiene `Male` y `Female`, mientras que `Dataset` solo contiene las clases 1 y 2. También se confirmaron 8 faltantes en `Gender` y 15 en `Dataset`.

Ambas columnas se convierten a categóricas para conservar sus etiquetas. En el caso de `Dataset`, la conversión usa explícitamente las categorías de texto `"1"` y `"2"`, que son las representaciones que tiene el archivo. Así se evita que los valores se conviertan por error en faltantes y se mantiene la variable como clasificación, no como booleano.

### Variables numéricas

In [ ]:
cols_numeric_float = [
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Protiens",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
]

numeric_conversion_summary_float = pd.DataFrame(
    {
        "no_nulos_antes": pacientes_df[cols_numeric_float].notna().sum(),
        "no_nulos_despues": pacientes_df[cols_numeric_float]
        .apply(pd.to_numeric, errors="coerce")
        .notna()
        .sum(),
    }
)
numeric_conversion_summary_float["convertidos_a_faltante"] = (
    numeric_conversion_summary_float["no_nulos_antes"]
    - numeric_conversion_summary_float["no_nulos_despues"]
)
print(numeric_conversion_summary_float)
pacientes_df[cols_numeric_float] = (
    pacientes_df[cols_numeric_float]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Float64")
)

In [ ]:
cols_numeric_int = ["Age"]

numeric_conversion_summary_int = pd.DataFrame(
    {
        "no_nulos_antes": pacientes_df[cols_numeric_int].notna().sum(),
        "no_nulos_despues": pacientes_df[cols_numeric_int]
        .apply(pd.to_numeric, errors="coerce")
        .notna()
        .sum(),
    }
)
numeric_conversion_summary_int["convertidos_a_faltante"] = (
    numeric_conversion_summary_int["no_nulos_antes"]
    - numeric_conversion_summary_int["no_nulos_despues"]
)
print(numeric_conversion_summary_int)
pacientes_df[cols_numeric_int] = (
    pacientes_df[cols_numeric_int].apply(pd.to_numeric, errors="coerce").astype("Int64")
)

### Hallazgos sobre tipos de datos

Las mediciones clínicas se reconocieron como variables numéricas y se convirtieron a `Float64`; `Age` se convirtió a `Int64`, que permite conservar su valor entero y también datos faltantes. Después de revisar las columnas numéricas, no se encontraron valores no numéricos adicionales. Por esta razón, `pd.to_numeric(..., errors="coerce")` no generó nuevos valores faltantes.

La conversión se hizo de forma explícita para que cada columna tenga un tipo adecuado. No se modificaron los valores clínicos ni se imputaron las ausencias.

### Valores faltantes y duplicados

In [ ]:
# Se conservan los valores faltantes para no introducir supuestos clínicos.
missing_after = pacientes_df.isna().sum().sort_values(ascending=False)
initial_row_count = len(pacientes_df)
duplicate_count = int(pacientes_df.duplicated().sum())
duplicate_rows = int(pacientes_df.duplicated(keep=False).sum())
duplicate_percentage = duplicate_count / initial_row_count
print("Valores faltantes después de la conversión por columna:")
print(missing_after[missing_after > 0])
print(f"Valores faltantes después de la conversión: {int(missing_after.sum())}")
print(f"Filas al comenzar la revisión de duplicados: {initial_row_count}")
print(f"Filas duplicadas exactas: {duplicate_count}")
print(f"Filas que pertenecen a grupos duplicados: {duplicate_rows}")
print(f"Proporción de filas duplicadas exactas: {duplicate_percentage:.2%}")

### Hallazgos sobre faltantes y duplicados

Después de unificar los valores vacíos y convertir los tipos, se mantienen **98 valores faltantes** en las 11 variables clínicas. Se conservan porque eliminarlos o completarlos requeriría una decisión clínica que no corresponde a esta etapa.

Se encontraron **60 filas completamente iguales**, equivalentes al **9,05 %** de las 663 filas. En total, 111 filas pertenecen a grupos con duplicados. El dataset no tiene un identificador de paciente, así que no es posible saber si las filas iguales corresponden al mismo paciente o a pacientes diferentes. En esta etapa, los duplicados se documentan como un hallazgo y no se eliminan. Su tratamiento se realizará más adelante, porque las observaciones repetidas pueden afectar los análisis posteriores y generar problemas de fuga de información si registros idénticos terminan distribuidos entre entrenamiento y evaluación.

In [ ]:
rows_after_duplicate_review = len(pacientes_df)
duplicates_after_review = int(pacientes_df.duplicated().sum())
print(f"Filas después de la revisión: {rows_after_duplicate_review}")
print(f"Duplicados exactos que se mantienen para una etapa posterior: {duplicates_after_review}")

### Resultado de la revisión de duplicados

Después de la revisión, el DataFrame conserva sus **663 filas** originales y los duplicados exactos siguen identificados, sin aplicar ninguna deduplicación. El tratamiento de estas filas se definirá en una etapa posterior para evitar que las repeticiones afecten los análisis o la separación entre entrenamiento y evaluación.

In [ ]:
pacientes_df.info()

In [ ]:
schema = pa.Table.from_pandas(pacientes_df, preserve_index=False).schema
schema

### 💾 Guardar el DataFrame tipado

In [ ]:
INTERMEDIATE_DIR = DATA_DIR / "02_intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = INTERMEDIATE_DIR / "Pacientes_porblemas_higado_india_type_fixed.parquet"

pacientes_df.to_parquet(
    PARQUET_PATH,
    index=False,
    engine="pyarrow",
)

### Validación del dataset final

In [ ]:
expected_dtypes = {
    "Age": "Int64",
    **{column: "Float64" for column in cols_numeric_float},
    "Gender": "category",
    "Dataset": "category",
}
assert list(pacientes_df.columns) == clinical_columns
assert pacientes_df.dtypes.astype(str).to_dict() == expected_dtypes
assert int(pacientes_df.isna().sum().sum()) == 98
assert PARQUET_PATH.exists()
reloaded_df = pd.read_parquet(PARQUET_PATH)
assert reloaded_df.equals(pacientes_df)
print(f"Dataset final validado y guardado en: {PARQUET_PATH.relative_to(PROJECT_ROOT)}")
print(f"Esquema validado: {reloaded_df.shape[0]} filas x {reloaded_df.shape[1]} columnas")

## 📊 Análisis de resultados

### Resumen de la exploración

- El archivo tiene 663 filas, 11 variables clínicas y 1.013 columnas `Unnamed:*`. Estas últimas contienen 21.273 valores no nulos en 21 filas, pero repiten los campos clínicos; por eso se excluyeron del dataset de trabajo.
- En todo el archivo se encontraron 650.444 cadenas vacías. Este total incluye las columnas auxiliares; en las 11 variables clínicas corresponden a 98 valores faltantes. No se encontraron `?`, `N/A`, `NA` ni `NaN`. Las representaciones se unificaron como `pd.NA`.
- Los 98 faltantes clínicos se mantienen sin imputación después de la conversión, porque no se deben inventar valores para completar las mediciones.
- `Gender` contiene únicamente `Male` y `Female` entre sus valores informados, y `Dataset` únicamente las clases 1 y 2. Las mediciones se tiparon como numéricas sin conversiones no numéricas adicionales.
- El dataset comenzó con 663 filas. Se identificaron 60 duplicados exactos, equivalentes al 9,05 %, y 111 filas pertenecientes a grupos con duplicados. Se mantienen en esta etapa y su tratamiento se realizará posteriormente.

El resultado conserva las 11 variables clínicas, mantiene los valores faltantes sin imputación y conserva los tipos de datos definidos durante la exploración. El dataset con sus 663 filas originales se guarda en formato Parquet.

## 💡 Propuestas e ideas

- Revisar posteriormente los valores extremos de las mediciones clínicas.

- Definir una estrategia de imputación únicamente después de analizar el contexto clínico y el objetivo del modelo.

## 📖 Referencias

- <https://archive.ics.uci.edu/static/public/225/ilpd+indian+liver+patient+dataset.zip>
- <https://pandas.pydata.org/docs/user_guide/pyarrow.html>